# 📐 이미지 데이터 연동 및 3D 공간 물리 분석 (OWL-ViT)

본 노트북은 비디오 프레임 순서 정렬 과제에서 **카메라 기법(줌인/줌아웃) 묘사 문장**과 **실제 이미지 속 피사체 크기 변화** 간의 시공간적 인과관계를 연동하여 VLM의 순서 정렬 정확도를 극대화하는 최종 구현 및 검증용 EDA 노트북입니다.

### 핵심 탑재 파이프라인:
1. **Gemma**: 문맥 내 카메라 기법(줌아웃 등) 및 추적 대상 사물 추출
2. **OWL-ViT**: Open-Vocabulary 사물 위치 검출 및 BBox 면적 비율 트렌드($R_{bbox}$) 측정
3. **Max Area Filter**: 동일 클래스 다중 객체 등장 시 메인 피사체 자동 고정
4. **VLM Soft Prompt**: 정량화된 수치 데이터를 자연스러운 형태의 시공간 힌트로 합성하여 Qwen2-VL에 주입

In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import torch

# PyTorch 중복 라이브러리 경고 방지
os.environ['KMP_DUPLICATE_LIB_OK'] = 'True'

try:
    from transformers import OwlViTProcessor, OwlViTForObjectDetection
    HAS_OWL = True
    print("✅ OWL-ViT (transformers) 로드 성공!")
except ImportError:
    HAS_OWL = False
    print("⚠️ 'transformers' 라이브러리가 없습니다. 설치하려면: !pip install transformers")

# OWL-ViT 라이브러리가 없을 때 데모 실행이 가능하도록 모의(Mock) 클래스 작성
if not HAS_OWL:
    class MockOwlViT:
        def __init__(self, *args, **kwargs):
            pass
        def eval(self):
            pass
        def to(self, device):
            return self
            
    class MockProcessor:
        @classmethod
        def from_pretrained(cls, name):
            return cls()
        def __call__(self, text, images, **kwargs):
            return {}
        def post_process_object_detection(self, outputs, target_sizes, threshold):
            h, w = target_sizes[0].tolist()
            # 모의 줌아웃 데이터 생성 (호출 순서에 따라 점점 작아지는 BBox)
            # 임의로 프레임마다 크기가 변하도록 모의
            mock_boxes = np.array([[w*0.2, h*0.2, w*0.8, h*0.8]])
            return [{
                "boxes": torch.tensor(mock_boxes),
                "scores": torch.tensor([0.95]),
                "labels": torch.tensor([0])
            }]
            
    OwlViTProcessor = MockProcessor
    OwlViTForObjectDetection = MockOwlViT
    print("ℹ️ 오프라인 구동 및 테스트를 위해 Mock OWL-ViT 모듈이 로드되었습니다.")

## 1. OWL-ViT 궤적 및 면적 추출기 정의

Open-Vocabulary 객체 검출 모델을 사용하여, 임의의 명사 `query_text`에 대해 4프레임에서의 BBox 면적 비율($R_{bbox} = \text{객체 면적} / \text{이미지 면적}$)과 중심 좌표($X, Y$)를 추출합니다.

In [ ]:
class OWLViTTrajectoryExtractor:
    def __init__(self, model_name="google/owlvit-base-patch32", device="cpu"):
        self.device = device
        # Mock 모듈과 실제 모듈을 호환하기 위해 try-catch 처리
        if HAS_OWL:
            self.processor = OwlViTProcessor.from_pretrained(model_name)
            self.model = OwlViTForObjectDetection.from_pretrained(model_name).to(self.device)
            self.model.eval()
        else:
            self.processor = OwlViTProcessor.from_pretrained(model_name)
            self.model = OwlViTForObjectDetection()

    def extract_object_trajectory(self, image_paths, query_text, threshold=0.10):
        """
        4장의 이미지 경로와 Open-Vocabulary 쿼리 텍스트를 받아 객체 위치 및 면적 변화를 추적합니다.
        """
        coords = []
        text_queries = [[query_text]]

        for idx, img_path in enumerate(image_paths):
            if not os.path.exists(img_path):
                # 데모용 이미지 미존재 시 모의 좌표 생성
                mock_areas = [0.342, 0.250, 0.185, 0.081]  # 줌아웃 추세 데모
                area_val = mock_areas[idx] if idx < len(mock_areas) else 0.15
                coords.append(
                    f"- Image {idx+1}: '{query_text}' center=[X={0.5+idx*0.01:.3f}, Y=0.500], Area={area_val*100:.1f}%"
                )
                continue

            img = Image.open(img_path).convert("RGB")
            w, h = img.size
            img_area = w * h

            # 실제 입력 처리
            if HAS_OWL:
                inputs = self.processor(text=text_queries, images=img, return_tensors="pt").to(self.device)
                with torch.no_grad():
                    outputs = self.model(**inputs)
                target_sizes = torch.tensor([img.size[::-1]], dtype=torch.float32).to(self.device)
                results = self.processor.post_process_object_detection(
                    outputs=outputs, target_sizes=target_sizes, threshold=threshold
                )[0]
                boxes = results["boxes"].cpu().numpy()
            else:
                # Mock 실행 시 결과
                results = self.processor.post_process_object_detection(None, [img.size[::-1]], threshold)[0]
                boxes = results["boxes"]

            if len(boxes) == 0:
                coords.append(f"- Image {idx+1}: no '{query_text}' detected (skip this cue)")
                continue

            # 면적이 가장 큰 BBox를 주 피사체로 일관되게 선택 (Max Area Filter)
            best_idx = 0
            max_area = 0
            for i, box in enumerate(boxes):
                box_w = box[2] - box[0]
                box_h = box[3] - box[1]
                area = box_w * box_h
                if area > max_area:
                    max_area = area
                    best_idx = i

            best_box = boxes[best_idx]
            x_center = ((best_box[0] + best_box[2]) / 2) / w
            y_center = ((best_box[1] + best_box[3]) / 2) / h
            best_area_ratio = max_area / img_area

            coords.append(
                f"- Image {idx+1}: '{query_text}' center=[X={x_center:.3f}, Y={y_center:.3f}], Area={best_area_ratio*100:.1f}%"
            )

        return coords

## 2. 시공간 보조 힌트 생성기

Gemma가 분류해낸 텍스트 기법 정보와 OWL-ViT가 수집한 이미지 면적 변화 통계를 조합하여, VLM(Qwen)이 최종적으로 읽게 될 힌트 프롬프트를 완성합니다.

In [ ]:
def build_vlm_auxiliary_hint(image_paths, sentence, query_text, camera_technique="Zoom-out"):
    """
    Gemma의 문장분석 기법과 OWL-ViT의 궤적을 엮어 VLM에 주입할 프롬프트 구조를 자동 생성합니다.
    """
    device = "cuda" if torch.cuda.is_available() else "cpu"
    extractor = OWLViTTrajectoryExtractor(device=device)
    
    # 4개 이미지에 대해 궤적 및 면적 추출
    measurements = extractor.extract_object_trajectory(image_paths, query_text)
    
    hints = []
    hints.append("[시각 분석 보조 시스템 힌트 (Visual-Textual Auxiliary Hints)]")
    hints.append("4장의 뒤섞인 이미지와 캡션을 대조하여 올바른 프레임 순서를 추론하세요.\n")
    
    # Gemma 단계 분석 결과 매핑
    hints.append(f"- 캡션 분석 가이드 (Gemma):")
    tech_guide = "주인공 피사체의 화면 면적(Area)이 점진적으로 작아지는 방향" if "out" in camera_technique.lower() else "주인공 피사체의 화면 면적(Area)이 점진적으로 커지는 방향"
    hints.append(f"  * 감지된 카메라 기법: {camera_technique} - {tech_guide}으로 배열되어야 함.")
    hints.append(f"  * 추적 대상 피사체: '{query_text}'\n")
    
    # OWL-ViT 단계 분석 결과 매핑
    hints.append(f"- 이미지 측정값 (OWL-ViT):")
    for m in measurements:
        hints.append(f"  {m}")
        
    # 최종 매핑 가이드 연산 (Area가 파싱 가능할 때 자동 정렬 표시)
    try:
        parsed_data = []
        for idx, m in enumerate(measurements):
            if "Area=" in m:
                area_val = float(m.split("Area=")[1].replace("%", ""))
                parsed_data.append((idx+1, area_val))
        
        if len(parsed_data) == 4:
            # 내림차순 정렬 (줌아웃 순서)
            sorted_data = sorted(parsed_data, key=lambda x: x[1], reverse=True)
            flow_str = " -> ".join([f"Image {img_id} ({area:.1f}%)" for img_id, area in sorted_data])
            
            hints.append(f"\n[최종 정렬 가이드]")
            hints.append(f"{camera_technique} 기법(Area 추세에 따름)을 충족하는 최적의 후보 순서는:")
            hints.append(f"{flow_str} 입니다.")
    except Exception as e:
        pass
        
    hints.append("이 흐름을 문맥과 대조하여 최종 셔플 인덱스 정답 [n, n, n, n]을 도출하세요.")
    
    return "\n".join(hints)

# 더미 입력으로 최종 프롬프트 형태 검증
fake_images = ["fake_1.jpg", "fake_2.jpg", "fake_3.jpg", "fake_4.jpg"]
caption_sample = "A kayak slowly shifts further from the lens, fading into the wide horizon."
print(build_vlm_auxiliary_hint(fake_images, caption_sample, "kayak", "Zoom-out"))

## 3. 실제 데이터셋 연동 실증 테스트

프로젝트 폴더 내 `train.csv`와 `data_train/` 이미지셋을 연결하여 실제 검출을 테스트해봅니다.

In [ ]:
csv_path = '../train.csv' if os.path.exists('../train.csv') else 'train.csv'
if os.path.exists(csv_path):
    df = pd.read_csv(csv_path)
    # 줌 또는 카메라 관련 문장이 있는 샘플 1개 검색
    camera_samples = df[df['Sentence'].str.contains('zoom|closer|further|camera|shot|lens|person', case=False, na=False)]
    
    if len(camera_samples) > 0:
        sample = camera_samples.iloc[0]
        print("--- 실제 데이터 샘플 로드 ---")
        print(f"Id: {sample['Id']}")
        print(f"Sentence: {sample['Sentence']}")
        print(f"Answer: {sample['Answer']}")
        
        img_dir = '../data_train' if os.path.exists('../data_train') else 'data_train'
        img_dir = os.path.join(img_dir, sample['Id'])
        
        if os.path.exists(img_dir):
            img_files = sorted(os.listdir(img_dir))
            full_paths = [os.path.join(img_dir, f) for f in img_files if f.lower().endswith(('.jpg', '.jpeg', '.png'))][:4]
            
            # 가상 쿼리 설정 (단어 검색 또는 기본 주인공 설정)
            query_word = "person"  # 디폴트 쿼리
            for word in ["kayak", "scissors", "comb", "car", "dog", "cat", "ball"]:
                if word in sample['Sentence'].lower():
                    query_word = word
                    break
            
            # 줌 인/아웃 방향 유추
            tech = "Zoom-out" if any(x in sample['Sentence'].lower() for x in ["further", "away", "zoom out", "zooms out"]) else "Zoom-in"
            
            # 최종 프롬프트 시뮬레이션 생성
            vlm_prompt = build_vlm_auxiliary_hint(full_paths, sample['Sentence'], query_word, tech)
            print("\n--- 합성된 VLM 프롬프트 템플릿 ---")
            print(vlm_prompt)
        else:
            print(f"\n⚠️ 이미지 폴더 '{img_dir}'를 찾지 못했습니다. 로컬 디스크 환경을 확인하세요.")
    else:
        print("적절한 카메라 관련 캡션을 가진 행을 찾지 못해 기본 행으로 대체합니다.")
else:
    print("⚠️ train.csv가 없습니다. 프로젝트 루트에서 구동하고 있는지 확인하세요.")